# Prepare inventories for flexpart folding
This notebook prepares the inventories required to fold with flexpart footprints
All the inventories used here are based on emiproc
emiproc can be installed with `pip install emiproc`


In [ ]:
%load_ext autoreload
# import 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from pathlib import Path

from matplotlib.colors import LogNorm, SymLogNorm

import cmcrameri.cm as cmc

import emiproc
from emiproc.inventories.edgar import download_edgar_files, EDGARv8
from emiproc.grids import RegularGrid
from emiproc.regrid import remap_inventory
from emiproc.plots import plot_inventory
from emiproc.inventories.wetcharts import WetCHARTs
from emiproc.exports.hourly import export_hourly_emissions
from emiproc.inventories.utils import group_categories
from emiproc.exports.rasters import export_raster_netcdf


# check which emiproc version is used
print(emiproc.__file__)

### Define where to save the downloaded inventory files


data_dir = Path("/project/leob/GAW/Kenya/data/") # on ddm
dir_out = Path("/newhome/leob/Documents/git-repos/gawkenya/analyses/output/inventories") # to save plots

use_nested = False # full or nested (Africa only) domain for flexpart
if use_nested == False:
    dir_out = dir_out / "full_domain"

download_edgar = False # need to download the files only once
export_files = False # set to True if you want to export the files, takes long time for all years
save_figs = True # use the emiproc functions to save the figures, they will not be shown then. To show them in the ntebook, set to False


### Downlodad inventory data

In [ ]:


# Download EDGAR files for the years 2020-2024

years = range(2020, 2023) # data available for 2020-2023
if download_edgar:
    for yr in years:
        local_dir = data_dir / f"EDGAR/{yr}"
        local_dir.mkdir(exist_ok=True)

        print(f"Downloading EDGAR {yr} CH4 files into {local_dir}")
        download_edgar_files(local_dir, year=yr, substances=["CH4"], version = "v2024") # latest GHG version: v2024
    
        print(f"Downloading EDGAR {yr} CO files")
        download_edgar_files(local_dir, year=yr, substances=["CO"], version = "AP_v8.1") # latest Air Pollutants (AP) version: AP_v8.1
    
        print(f"Downloading EDGAR {yr} black carbon files")
        download_edgar_files(local_dir, year=yr, substances=["BC"], version = "AP_v8.1") # latest Air Pollutants (AP) version: AP_v8.1
    

In [ ]:
## Reading in the downloaded files as inventories
# Wetcharts has to be downloaded separately
yr = 2020
#for yr in years:
local_dir = data_dir / f"EDGAR/{yr}"

inv_edgar_ch4 = EDGARv8(local_dir / f"EDGAR_2024_GHG_CH4_{yr}_*.nc")

inv_edgar_co = EDGARv8(local_dir / f"v8.1_FT2022_AP_CO_{yr}*.nc")

inv_edgar_bc = EDGARv8(local_dir / f"v8.1_FT2022_AP_BC_{yr}*.nc")

# Wetcharts: 
wetchart_path = Path("/project/leob/GAW/Kenya/data/WetCHARTs/")
file_path =  wetchart_path / f"WetCHARTs_v1_3_3_{yr}.nc"
inv_wetcharts_ch4 = WetCHARTs(
    file_path,
    model=None, #use mean of all models
    category="wetland_emissions",
)

### Remap to Flexpart-domain (Africa)

In [ ]:
## Remap the inventory to flexpart grid

# decide whether to use the Africa only grid or the full domain grid

# The grid can be defined by various parameters
# See the documentation for more details
# https://emiproc.readthedocs.io/en/master/api/grids.html#emiproc.grids.RegularGrid

if use_nested:
    #african_grid = RegularGrid(xmin=0.05, xmax=60.05, ymin=-34.05, ymax=20.05, dx=0.1, dy=0.1) # 0.1 degree grid used in flexpart for Africa
    african_grid = RegularGrid(xmin=0, xmax=60.05, ymin=-35, ymax=20.05, dx=0.1, dy=0.1) # 0.1 degree grid used in flexpart for Africa

    # Remap the inventory on the grid
    inv_edgar_ch4_africa = remap_inventory(inv_edgar_ch4, african_grid)
    inv_edgar_co_africa = remap_inventory(inv_edgar_co, african_grid)
    inv_edgar_bc_africa = remap_inventory(inv_edgar_bc, african_grid)
    inv_wetcharts_ch4_africa = remap_inventory(inv_wetcharts_ch4, african_grid)
else: 
    # same but full Flexpart domain (Africa and Indian Ocean)
    #african_grid = RegularGrid(xmin=-21.25, xmax=111.25, ymin=-40.25, ymax=50.25, dx=0.5, dy=0.5) # 0.5 degree grid used in flexpart for full domain
    african_grid = RegularGrid(xmin=-20, xmax=110.5, ymin=-40, ymax=50.5, dx=0.5, dy=0.5) # 0.5 degree grid used in flexpart for full domain. When exporting to netcdf, the center-gridpoints will be used as coordinates, so add half a grid to the desired one at both edges!

    # Remap the inventory on the grid
    inv_edgar_ch4_africa = remap_inventory(inv_edgar_ch4, african_grid)
    inv_edgar_co_africa = remap_inventory(inv_edgar_co, african_grid)
    inv_edgar_bc_africa = remap_inventory(inv_edgar_bc, african_grid)
    inv_wetcharts_ch4_africa = remap_inventory(inv_wetcharts_ch4, african_grid)

In [ ]:
plot_inventory(inv_wetcharts_ch4_africa, total_only=True, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/CH4/all_sectors" if save_figs else None)
plot_inventory(inv_edgar_ch4_africa, total_only=True, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/CH4/all_sectors" if save_figs else None)
plot_inventory(inv_edgar_co_africa, total_only=True, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/CO/all_sectors" if save_figs else None)
plot_inventory(inv_edgar_bc_africa, total_only=True, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/BC/all_sectors" if save_figs else None)

### Group source categories

In [ ]:
# categories for GHG
ghg_categories = {
        "agriculture": [
            "Agricultural soils",
            "Agricultural waste burning",
            "Manure management",
            "Enteric fermentation"
        ],
        "energy": [
            "Power Industry",
            "Fuel exploitation",
            "Energy for buildings",
            ],
        "industry": [
            "Oil refineries and Transformation industry",
            "Chemical processes",
            "Combustion for manufacturing",
            "Iron and steel production",
            #"Non energy use of fuels", 
            #"Solvents and products use",
            #"Non-ferrous metals production", #not for GHG
            #"Non-metallic minerals production", #not for GHG
        ],
        "waste": [
            "Waste water handling",
            "Solid waste incineration",
            "Solid waste landfills",
        ],
        "transportation": [
            "Aviation climbing_and_descent",
            "Aviation cruise",
            "Aviation landing_and_takeoff",
            "Railways, pipelines, off-road transport",
            "Shipping",
            "Road transportation",
        ],
    }

In [ ]:
# categories for air pollutants
ap_categories = {
        "agriculture": [
            #"Agricultural soils",
            "Agricultural waste burning",
            #"Manure management",
        ],
        "energy": [
            "Power Industry",
            "Fuel exploitation",
            "Energy for buildings",
            ],
        "industry": [
            "Oil refineries and Transformation industry",
            "Chemical processes",
            "Combustion for manufacturing",
            "Iron and steel production",
            "Non-ferrous metals production", # only for AP
            "Non-metallic minerals production", # only for AP
        ],
        #"livestock": ["Enteric fermentation"], # not for AP
        "waste": [
        #     "Waste water handling", # not for AP
             "Solid waste incineration",
        #     "Solid waste landfills", # not for AP
        ],
        "transportation": [
            "Aviation climbing_and_descent",
            "Aviation cruise",
            "Aviation landing_and_takeoff",
            "Railways, pipelines, off-road transport",
            "Shipping",
            "Road transportation",
        ],
    }

In [ ]:
inv_edgar_ch4_africa_grouped = group_categories(inv_edgar_ch4_africa, categories_group = ghg_categories)
inv_edgar_co_africa_grouped = group_categories(inv_edgar_co_africa, categories_group = ap_categories)
inv_edgar_bc_africa_grouped = group_categories(inv_edgar_bc_africa, categories_group = ap_categories)

In [ ]:
# Look at totals of new groups
plot_inventory(inv_edgar_ch4_africa_grouped, total_only=True, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/CH4" if save_figs else None)  #give out_dir to save the plots
plot_inventory(inv_edgar_co_africa_grouped, total_only=True, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/CO" if save_figs else None)
plot_inventory(inv_edgar_bc_africa_grouped, total_only=True, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/BC" if save_figs else None)

In [ ]:
# Look at all grouped sector's distribution
plot_inventory(inv_edgar_ch4_africa_grouped, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/CH4")
plot_inventory(inv_edgar_co_africa_grouped, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/CO")
plot_inventory(inv_edgar_bc_africa_grouped, cmap = "cmc.davos_r", figsize=(18,5), out_dir = dir_out/"emiproc/BC")

### Export the remapped and regrouped files

In [ ]:

if export_files:
    if use_nested:
        export_path = data_dir / "EDGAR/EDGAR_africa_grouped/"
    else:
        export_path = data_dir / "EDGAR/EDGAR_africa_india_grouped/"
    export_path.mkdir(exist_ok=True)
    export_raster_netcdf(inv_edgar_ch4_africa_grouped, export_path / f"CH4_{yr}.nc")
    export_raster_netcdf(inv_edgar_co_africa_grouped, export_path / f"CO_{yr}.nc")
    export_raster_netcdf(inv_edgar_bc_africa_grouped, export_path / f"BC_{yr}.nc")

### For wetcharts, export monthly files

In [ ]:
#
if export_files:
    monthly_dir = wetchart_path / "monthly"
    monthly_dir.mkdir(parents=True, exist_ok=True)
    # saves monthly netcdfs in monthly_dir
    export_hourly_emissions(
        inv_wetcharts_ch4_africa,
        path=monthly_dir,
        freq="MS", # Start of the month
    )


In [ ]:
export_files = True

## Do the same as all above but for each year at once

In [ ]:
# this takes long time
# make sure to define the sector groups before

read_wetcharts = True
read_edgar = False
use_nested = False # full or nested (Africa only) domain for flexpart

if use_nested:
    african_grid = RegularGrid(xmin=0, xmax=60.05, ymin=-35, ymax=20.05, dx=0.1, dy=0.1) # 0.1 degree grid used in flexpart for Africa
else: 
    # same but full Flexpart domain (Africa and Indian Ocean)
    #african_grid = RegularGrid(xmin=-21.25, xmax=111.25, ymin=-40.25, ymax=50.25, dx=0.5, dy=0.5) # 0.5 degree grid used in flexpart for full domain
    african_grid = RegularGrid(xmin=-20, xmax=110.5, ymin=-40, ymax=50.5, dx=0.5, dy=0.5) # 0.5 degree grid used in flexpart for full domain. When exporting to netcdf, the center-gridpoints will be used as coordinates, so add half a grid to the desired one at both edges!

if export_files:
    wetchart_path = Path("/project/leob/GAW/Kenya/data/WetCHARTs/")
    if use_nested:
        export_path = data_dir / "EDGAR/EDGAR_africa_grouped/"
    else:
        export_path = data_dir / "EDGAR/EDGAR_africa_india_grouped/"
    monthly_dir = wetchart_path / "monthly"
    last_year_wetchart = 2021
    last_year_AP = 2022

    years = range(2020, 2024) # data available for 2020-2023
    for yr in years:
        local_dir = data_dir / f"EDGAR/{yr}"
        
        ## Reading in the downloaded files as inventories
        if read_edgar:
            print(f"Reading in EDGAR {yr} files")
            inv_edgar_ch4 = EDGARv8(local_dir / f"EDGAR_2024_GHG_CH4_{yr}_*.nc")

            if yr <= last_year_AP: 
                inv_edgar_co = EDGARv8(local_dir / f"v8.1_FT2022_AP_CO_{yr}*.nc")

                inv_edgar_bc = EDGARv8(local_dir / f"v8.1_FT2022_AP_BC_{yr}*.nc")

        # Wetcharts: 
        if read_wetcharts:
            if yr <= last_year_wetchart: 
                print(f"Reading in WetCHARTs {yr} files")
                file_path =  wetchart_path / f"WetCHARTs_v1_3_3_{yr}.nc"
                inv_wetcharts_ch4 = WetCHARTs(
                    file_path,
                    model=None, #use mean of all models
                    category="wetland_emissions",
                )

        ### Remap the inventory on the grid
        if read_edgar:
            print(f"Remapping EDGAR {yr} files")
            inv_edgar_ch4_africa = remap_inventory(inv_edgar_ch4, african_grid)
            if yr <= last_year_AP: 
                inv_edgar_co_africa = remap_inventory(inv_edgar_co, african_grid)
                inv_edgar_bc_africa = remap_inventory(inv_edgar_bc, african_grid)
        if read_wetcharts:
            print(f"Remapping wetcharts {yr} files")
            if yr <= last_year_wetchart: 
                inv_wetcharts_ch4_africa = remap_inventory(inv_wetcharts_ch4, african_grid)

        ### Group the categories
        if read_edgar:
            inv_edgar_ch4_africa_grouped = group_categories(inv_edgar_ch4_africa, categories_group = ghg_categories)
            if yr <= last_year_AP: 
                inv_edgar_co_africa_grouped = group_categories(inv_edgar_co_africa, categories_group = ap_categories)
                inv_edgar_bc_africa_grouped = group_categories(inv_edgar_bc_africa, categories_group = ap_categories)

        ### Export the remapped and grouped inventories
        if read_edgar:
            export_raster_netcdf(inv_edgar_ch4_africa_grouped, export_path / f"CH4_{yr}.nc")
            if yr <= last_year_AP: 
                export_raster_netcdf(inv_edgar_co_africa_grouped, export_path / f"CO_{yr}.nc")
                export_raster_netcdf(inv_edgar_bc_africa_grouped, export_path / f"BC_{yr}.nc")
            print(f"Exported {yr} files to {export_path}")

        # For wetcharts, export monthly files
        if read_wetcharts:
            if yr <= last_year_wetchart:
                # attention, in 2022 wetcharts data stops in August!
                if yr == 2022:
                    end_time = pd.Timestamp(f"{inv_wetcharts_ch4_africa.year}-08-01 00:00:00") 
                else:
                    end_time = None #take full year

                
                            
                if use_nested:
                    monthly_dir = wetchart_path / "monthly"
                else:
                    monthly_dir = wetchart_path / "monthly_africa_india"
                monthly_dir.mkdir(parents=True, exist_ok=True)
                # saves monthly netcdfs in monthly_dir
                export_hourly_emissions(
                    inv_wetcharts_ch4_africa,
                    path=monthly_dir,
                    freq="MS", # Start of the month
                    end_time = end_time, # only for 2022
                )
                print(f"Exported WetCHARTs {yr} files to {monthly_dir}")

In [ ]:
xr.open_dataset("/project/leob/GAW/Kenya/data/WetCHARTs/monthly_africa_india/20200301T000000Z.nc")

In [ ]:
xr.open_dataset("/project/leob/GAW/Kenya/data/EDGAR/EDGAR_africa_india_grouped/BC_2022.nc")

In [ ]:
test = xr.open_dataset("/project/leob/GAW/Kenya/data/EDGAR/EDGAR_africa_grouped/BC_2021.nc")
test

In [ ]:
#convert kg/h/cell to kg/m2/s

test["CH4_wetland_emissions"] / 3600 / test["cell_area"]  # kg/h/cell to kg/m2/s

In [ ]:
plt.figure()
(test["CH4_wetland_emissions"] / (3600) / test["cell_area"] ).plot()
plt.show()

In [ ]:
plt.figure()
(test["CH4_wetland_emissions"] ).plot()
plt.show()